# NUST Bank Assistant – QLoRA Fine-Tuning with UnSloth

This notebook fine-tunes **Qwen2.5-3B-Instruct** on the NUST Bank product knowledge dataset using **UnSloth** (4-bit QLoRA).

**Requirements:** Run on Google Colab with a **T4 GPU** (free tier works).

## Steps
1. Install dependencies
2. Load & prepare training data from the bank dataset
3. Load Qwen2.5-3B-Instruct with UnSloth (4-bit quantization)
4. Fine-tune with QLoRA via SFTTrainer
5. Test inference
6. Save the LoRA adapter

## 1. Install Dependencies

In [1]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes xformers

## 2. Upload & Prepare Training Data

Upload `cleaned_chunks.json` (generated by `src/data_pipeline.py`) to Colab, then convert to instruction-tuning format.

In [2]:
import json
from google.colab import files

# Upload cleaned_chunks.json
print("Upload cleaned_chunks.json:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Upload cleaned_chunks.json:


Saving cleaned_chunks.json to cleaned_chunks.json
Loaded 303 chunks


In [3]:
# Convert chunks to instruction-tuning format
# For chunks that are Q&A pairs, extract question/answer
# For others, create a "summarize/explain" instruction

training_data = []

system_msg = (
    "You are a professional customer service assistant for NUST Bank. "
    "Answer questions about NUST Bank products and services accurately and politely. "
    "Only provide information from the bank's official documents. "
    "Never give financial advice."
)

for chunk in chunks:
    text = chunk["text"]
    meta = chunk.get("metadata", {})
    source = meta.get("sheet", meta.get("category", meta.get("source", "")))

    # Check if chunk is a Q&A pair
    if text.startswith("Q:") and "\nA:" in text:
        parts = text.split("\nA:", 1)
        question = parts[0].replace("Q:", "").strip()
        answer = parts[1].strip()
        if "\nNotes:" in answer:
            ans_parts = answer.split("\nNotes:", 1)
            answer = ans_parts[0].strip() + " " + ans_parts[1].strip()
    else:
        # Create a question about this information
        question = f"What can you tell me about {source}?" if source else "What information do you have about this bank product?"
        answer = text

    if len(question.strip()) < 5 or len(answer.strip()) < 5:
        continue

    training_data.append({
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    })

print(f"Created {len(training_data)} training examples")
print("\nSample:")
for msg in training_data[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}")

Created 301 training examples

Sample:
  [system]: You are a professional customer service assistant for NUST Bank. Answer questions about NUST Bank pr
  [user]: Is there a limit on the amount I can transfer through the mobile banking app?
  [assistant]: Yes, 1 million is the current daily limit. Transfer limits vary based on your account type. Check th


## 3. Load Model with UnSloth (4-bit Quantization)

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit
Parameters: 3,085,938,688


In [5]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,          # optimized – 0 is faster
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves 30% VRAM
    random_state=42,
)

model.print_trainable_parameters()

Unsloth 2026.3.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 4. Prepare Dataset for SFTTrainer

In [6]:
from datasets import Dataset

# Convert to HF Dataset
def format_chat(example):
    """Apply the chat template to produce the final training text."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_chat)

print(f"Dataset size: {len(dataset)}")
print(f"\nSample formatted text (first 500 chars):\n{dataset[0]['text'][:500]}")

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Dataset size: 301

Sample formatted text (first 500 chars):
<|im_start|>system
You are a professional customer service assistant for NUST Bank. Answer questions about NUST Bank products and services accurately and politely. Only provide information from the bank's official documents. Never give financial advice.<|im_end|>
<|im_start|>user
Is there a limit on the amount I can transfer through the mobile banking app?<|im_end|>
<|im_start|>assistant
Yes, 1 million is the current daily limit. Transfer limits vary based on your account type. Check the "Limits


## 5. Fine-Tune with QLoRA

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/301 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 301 | Num Epochs = 3 | Total steps = 57
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
10,3.798027
20,1.699142
30,1.521607
40,1.353730
50,1.258502



Training complete!
  Total steps: 57
  Training loss: 1.8502


## 6. Test Inference

In [8]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

test_questions = [
    "What is the Little Champs Account?",
    "How can I open a Roshan Digital Account?",
    "What are the profit rates for savings accounts?",
    "How do I transfer funds using the mobile app?",
    "Tell me a joke",  # out-of-domain test
]

system_msg = (
    "You are a professional customer service assistant for NUST Bank. "
    "Answer questions about NUST Bank products and services accurately and politely. "
    "Only provide information from the bank's official documents. "
    "Never give financial advice."
)

for q in test_questions:
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": q},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]

    outputs = model.generate(
        input_ids=input_ids, max_new_tokens=256, temperature=0.7, do_sample=True
    )
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_c


Q: What is the Little Champs Account?
A: NUST Sahar Bank's Little Champ's Account is designed to cater to the needs of the young and growing minds. It offers low maintenance charges, free internet banking, and cashless transactions through NFC-enabled debit cards.
------------------------------------------------------------

Q: How can I open a Roshan Digital Account?
A: Roshan Digital Account can be opened by:
-Existing NUST Sahar Current Account holders with minimum Rs. 10,000/- in their account
-Existing NUST Sahar Savings Account holders with minimum Rs. 10,000/- in their account
-Existing NUST Sahar Finance Account holders with minimum Rs. 10,000/- in their account
-New customers of NUST Sahar Current Account with minimum Rs. 10,000/- in their account
-New customers of NUST Sahar Finance Account with minimum Rs. 10,000/- in their account
------------------------------------------------------------

Q: What are the profit rates for savings accounts?
A: Profit Rates will be as per 

## 7. Save LoRA Adapter

In [9]:
# Save locally
model.save_pretrained("nust_bank_lora_adapter")
tokenizer.save_pretrained("nust_bank_lora_adapter")
print("LoRA adapter saved to nust_bank_lora_adapter/")

# Download the adapter files
import shutil
shutil.make_archive("nust_bank_lora_adapter", "zip", "nust_bank_lora_adapter")
files.download("nust_bank_lora_adapter.zip")
print("Download started – save this adapter to use in the RAG pipeline.")

LoRA adapter saved to nust_bank_lora_adapter/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started – save this adapter to use in the RAG pipeline.


## 8. (Optional) Push to Hugging Face Hub

In [ ]:
# Uncomment and fill in your details to push to HF Hub
# model.push_to_hub("your-username/nust-bank-qwen2.5-3b-lora", token="hf_...")
# tokenizer.push_to_hub("your-username/nust-bank-qwen2.5-3b-lora", token="hf_...")